# 🌾 Otimização de Recursos Agrícolas via Algoritmos Genéticos
### **FarmTech Solutions - Fase 7 / Desafio Ir Além (Seção 3.2)**

**Alunos:** Leonardo de Mattos Oliveira (RM 567749) e Gabriela de Andrade Alves (RM 567740)

Este notebook documenta a implementação de um **Algoritmo Genético (AG)** para resolver o problema de alocação eficiente de insumos (recursos hídricos e adubo) na fazenda da FarmTech Solutions, comparando diferentes heurísticas estruturantes de seleção, cruzamento (crossover) e mutação em termos de tempo de execução e qualidade do resultado (aptidão final).

## 1. Formulação Matemática do Problema

O problema é modelado como uma **Mochila Binária Multidimensional**:
- Temos $N$ talhões (lotes de terra) de plantio, cada um necessitando de uma quantidade de recursos e gerando um rendimento esperado.
- Seja $x_i \in \{0, 1\}$ a variável de decisão, onde $x_i = 1$ indica que o talhão $i$ receberá insumos e será cultivado, e $x_i = 0$ indica que o talhão não será ativado.
- $v_i$ é o rendimento financeiro esperado do talhão $i$.
- $w_i$ é a demanda de água do talhão $i$.
- $f_i$ é a demanda de adubo do talhão $i$.
- $W_{max}$ é o orçamento hídrico total diário da fazenda.
- $F_{max}$ é o orçamento de adubo total diário da fazenda.

**Função Objetivo (Maximizar o Rendimento Total):**
$$\text{Maximizar } Z = \sum_{i=1}^{N} v_i \cdot x_i$$

**Sujeito a (Restrições de Recursos):**
$$\sum_{i=1}^{N} w_i \cdot x_i \le W_{max} \quad (\text{Restrição de Água})$$
$$\sum_{i=1}^{N} f_i \cdot x_i \le F_{max} \quad (\text{Restrição de Adubo})$$
$$x_i \in \{0, 1\}, \quad \forall i \in \{1, \dots, N\}$$

### Função de Penalização Externa
Para lidar com cromossomos que violam as restrições, aplicamos uma função de aptidão com penalidade:
$$\text{Fitness}(X) = \sum_{i=1}^{N} v_i \cdot x_i - \text{Penalidade}(X)$$
Onde a $\text{Penalidade}(X) = 0$ se as restrições forem atendidas, e cresce proporcionalmente ao excesso caso contrário.

## 2. Preparação do Ambiente e Leitura Reprodutiva dos Dados

In [ ]:
import os
import json
import random
import time
import numpy as np
import matplotlib.pyplot as plt

# Carregar dados salvos em ga_data.json para garantir reprodutibilidade completa
data_path = 'ga_data.json'
if not os.path.exists(data_path):
    # Caso rode isolado fora da pasta, cria de forma consistente
    culturas = ["Soja", "Milho", "Trigo", "Algodão", "Arroz"]
    random.seed(42)
    plots = []
    for i in range(1, 16):
        plots.append({
            "id": i,
            "nome": f"Talhao {i} ({random.choice(culturas)})",
            "demanda_agua": random.randint(1500, 6000),
            "demanda_adubo": random.randint(80, 300),
            "rendimento_estimado": random.randint(5000, 30000)
        })
    with open(data_path, 'w', encoding='utf-8') as f:
        json.dump(plots, f, indent=4)
else:
    with open(data_path, 'r', encoding='utf-8') as f:
        plots = json.load(f)

print(f"Dados carregados. Numero de talhoes: {len(plots)}")
print("Exemplo do primeiro talhão:", plots[0])

## 3. Implementação do Algoritmo Genético

In [ ]:
class GeneticAlgorithm:
    def __init__(self, plots, max_water, max_fertilizer, 
                 pop_size=50, generations=100, mutation_rate=0.05, crossover_rate=0.8,
                 selection_method="tournament", crossover_method="single_point", mutation_method="bit_flip"):
        self.plots = plots
        self.max_water = max_water
        self.max_fertilizer = max_fertilizer
        self.pop_size = pop_size
        self.generations = generations
        self.mutation_rate = mutation_rate
        self.crossover_rate = crossover_rate
        self.selection_method = selection_method
        self.crossover_method = crossover_method
        self.mutation_method = mutation_method
        self.num_genes = len(plots)
        
    def calculate_fitness(self, chromosome):
        total_water = 0
        total_fertilizer = 0
        total_yield = 0
        for i, gene in enumerate(chromosome):
            if gene == 1:
                total_water += self.plots[i]['demanda_agua']
                total_fertilizer += self.plots[i]['demanda_adubo']
                total_yield += self.plots[i]['rendimento_estimado']
        
        # Restrições
        if total_water > self.max_water or total_fertilizer > self.max_fertilizer:
            excesso_agua = max(0, total_water - self.max_water)
            excesso_adubo = max(0, total_fertilizer - self.max_fertilizer)
            return max(1, total_yield - (100 * excesso_agua + 500 * excesso_adubo))
        return total_yield

    # Seleções
    def selection(self, population):
        if self.selection_method == "tournament":
            selected = random.sample(population, 3)
            fits = [self.calculate_fitness(ind) for ind in selected]
            return selected[fits.index(max(fits))]
        else: # Roulette
            fits = [self.calculate_fitness(ind) for ind in population]
            total = sum(fits)
            if total == 0: return random.choice(population)
            pick = random.uniform(0, total)
            curr = 0
            for i, ind in enumerate(population):
                curr += fits[i]
                if curr > pick:
                    return ind
            return population[-1]

    # Crossovers
    def crossover(self, p1, p2):
        if random.random() > self.crossover_rate:
            return p1.copy(), p2.copy()
        if self.crossover_method == "single_point":
            point = random.randint(1, self.num_genes - 1)
            return p1[:point] + p2[point:], p2[:point] + p1[point:]
        else: # Uniform
            c1, c2 = [], []
            for i in range(self.num_genes):
                if random.random() < 0.5:
                    c1.append(p1[i]); c2.append(p2[i])
                else:
                    c1.append(p2[i]); c2.append(p1[i])
            return c1, c2

    # Mutações
    def mutate(self, ind):
        mutated = ind.copy()
        if self.mutation_method == "bit_flip":
            for i in range(self.num_genes):
                if random.random() < self.mutation_rate:
                    mutated[i] = 1 - mutated[i]
        else: # Swap
            if random.random() < self.mutation_rate:
                i1, i2 = random.sample(range(self.num_genes), 2)
                mutated[i1], mutated[i2] = mutated[i2], mutated[i1]
        return mutated

    def run(self):
        start = time.time()
        # População inicial
        pop = [[random.randint(0, 1) for _ in range(self.num_genes)] for _ in range(self.pop_size)]
        best_ind = None
        best_fit = -1
        history = []
        
        for gen in range(self.generations):
            fits = [self.calculate_fitness(ind) for ind in pop]
            cur_best_idx = fits.index(max(fits))
            if fits[cur_best_idx] > best_fit:
                best_fit = fits[cur_best_idx]
                best_ind = pop[cur_best_idx].copy()
            
            history.append(best_fit)
            
            # Elitismo + nova população
            new_pop = [pop[cur_best_idx].copy()]
            while len(new_pop) < self.pop_size:
                p1 = self.selection(pop)
                p2 = self.selection(pop)
                c1, c2 = self.crossover(p1, p2)
                new_pop.extend([self.mutate(c1), self.mutate(c2)])
            pop = new_pop[:self.pop_size]
            
        return best_ind, best_fit, history, time.time() - start

## 4. Comparação de Estratégias (Torneio vs Roleta, Uniforme vs Ponto Único)

In [ ]:
W_max = 25000
F_max = 1200

# Executar Estratégia A: Torneio + Crossover Uniforme + Mutação Bit-Flip
ga_a = GeneticAlgorithm(plots, W_max, F_max, pop_size=60, generations=100, mutation_rate=0.08,
                        selection_method="tournament", crossover_method="uniform", mutation_method="bit_flip")
best_ind_a, best_fit_a, hist_a, time_a = ga_a.run()

# Executar Estratégia B: Roleta + Crossover Ponto Único + Mutação Swap
ga_b = GeneticAlgorithm(plots, W_max, F_max, pop_size=60, generations=100, mutation_rate=0.08,
                        selection_method="roulette", crossover_method="single_point", mutation_method="swap")
best_ind_b, best_fit_b, hist_b, time_b = ga_b.run()

print("=== RESULTADOS COMPARAÇÃO ===")
print(f"Estratégia A (Torneio+Uniforme+BitFlip): Fitness = R$ {best_fit_a:.2f} | Tempo = {time_a:.4f}s")
print(f"Estratégia B (Roleta+PontoUnico+Swap): Fitness = R$ {best_fit_b:.2f} | Tempo = {time_b:.4f}s")

## 5. Visualização das Curvas de Convergência

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(hist_a, label=f'Estratégia A (Torneio) - Melhor: R$ {best_fit_a}', color='green', linewidth=2)
plt.plot(hist_b, label=f'Estratégia B (Roleta) - Melhor: R$ {best_fit_b}', color='orange', linestyle='--', linewidth=2)
plt.title('Curvas de Convergência do Fitness ao Longo das Gerações')
plt.xlabel('Geração')
plt.ylabel('Melhor Fitness (R$)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 6. Conclusões Críticas sobre os Operadores

1. **Tempo de Execução:** A **Estratégia A (Torneio)** é significativamente mais rápida que a **Estratégia B (Roleta)**. Isso ocorre porque a seleção por roleta exige o cálculo da soma cumulativa e sorteios proporcionais de ponto flutuante em cima de toda a população a cada iteração, enquanto o torneio faz apenas buscas locais indexadas em pequenos subconjuntos ($k=3$), possuindo complexidade computacional inferior.
2. **Qualidade do Resultado (Convergência):** A **Estratégia A** também se mostra superior na velocidade de convergência e estabilidade. O cruzamento uniforme e a mutação por Bit-Flip garantem uma diversidade genética maior e evitam a convergência prematura para mínimos locais, algo comum na roleta devido à perda rápida de pressão seletiva quando indivíduos de média aptidão começam a dominar a roleta.